# Bangla License Plate Deblurring - Modified DeblurGAN-v2
Generator upgrade: **Line-Curve blocks** (horizontal strip conv + vertical strip conv + deformable conv) so the network focuses on the straight lines and curves of Bangla letters and digits.

## Libraries

In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from torchvision.ops import DeformConv2d
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

device = torch.device("cuda")
torch.backends.cudnn.benchmark = True
print("GPU:", torch.cuda.get_device_name(0))

## Paths and settings

In [ ]:
CLEAN_DIR = r"D:\academy\thesis\dataset\thesis dataset\de-blur gan line curve\input\clean_plate"  # TODO: set your local path
DEFOCUS_DIR = r"D:\academy\thesis\dataset\thesis dataset\de-blur gan line curve\input\defocus_blurred"  # TODO: set your local path
MOTION_DIR = r"D:\academy\thesis\dataset\thesis dataset\de-blur gan line curve\input\motion_blurred"  # TODO: set your local path

IMG_H = 128
IMG_W = 256
BATCH_SIZE = 8
EPOCHS = 10
LR = 1e-4
VAL_SPLIT = 0.1
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

## Dataset (images are loaded into RAM once, so training is faster)

In [ ]:
valid_ext = (".png", ".jpg", ".jpeg", ".bmp")

clean_names = sorted([f for f in os.listdir(CLEAN_DIR) if f.lower().endswith(valid_ext)])
random.shuffle(clean_names)

val_count = int(len(clean_names) * VAL_SPLIT)
val_names = clean_names[:val_count]
train_names = clean_names[val_count:]


def make_pairs(names):
    pairs = []
    for name in names:
        for blur_dir in (DEFOCUS_DIR, MOTION_DIR):
            blur_path = os.path.join(blur_dir, name)
            if os.path.exists(blur_path):
                pairs.append((blur_path, os.path.join(CLEAN_DIR, name)))
    return pairs


class PlateDataset(Dataset):
    def __init__(self, pairs):
        self.blur = []
        self.clean = []
        for blur_path, clean_path in tqdm(pairs):
            self.blur.append(self.load(blur_path))
            self.clean.append(self.load(clean_path))

    def __len__(self):
        return len(self.blur)

    def load(self, path):
        img = Image.open(path).convert("RGB").resize((IMG_W, IMG_H), Image.BICUBIC)
        return torch.from_numpy(np.asarray(img).copy()).permute(2, 0, 1)

    def __getitem__(self, i):
        return self.blur[i].float() / 127.5 - 1, self.clean[i].float() / 127.5 - 1


train_pairs = make_pairs(train_names)
val_pairs = make_pairs(val_names)

train_loader = DataLoader(PlateDataset(train_pairs), batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True, drop_last=True)
val_loader = DataLoader(PlateDataset(val_pairs), batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

print("clean images:", len(clean_names))
print("train pairs:", len(train_pairs))
print("val pairs:", len(val_pairs))

## Generator building blocks (the modified part)

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, stride, 1),
            nn.InstanceNorm2d(out_ch),
            nn.ReLU(True),
        )

    def forward(self, x):
        return self.body(x)


class LineCurveBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.horizontal = nn.Conv2d(ch, ch, (1, 7), padding=(0, 3))
        self.vertical = nn.Conv2d(ch, ch, (7, 1), padding=(3, 0))

        self.offset = nn.Conv2d(ch, 18, 3, padding=1)
        nn.init.zeros_(self.offset.weight)
        nn.init.zeros_(self.offset.bias)
        self.curve = DeformConv2d(ch, ch, 3, padding=1)

        self.fuse = nn.Sequential(
            nn.Conv2d(ch * 3, ch, 1),
            nn.InstanceNorm2d(ch),
            nn.ReLU(True),
        )
        self.gate = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(ch, ch // 4, 1),
            nn.ReLU(True),
            nn.Conv2d(ch // 4, ch, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        h = self.horizontal(x)
        v = self.vertical(x)
        c = self.curve(x, self.offset(x))
        y = self.fuse(torch.cat([h, v, c], dim=1))
        y = y * self.gate(y)
        return x + y

## Generator

In [ ]:
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.head = nn.Sequential(
            nn.Conv2d(3, 64, 7, padding=3),
            nn.InstanceNorm2d(64),
            nn.ReLU(True),
        )
        self.enc1 = nn.Sequential(LineCurveBlock(64), LineCurveBlock(64))
        self.down1 = ConvBlock(64, 128, stride=2)
        self.enc2 = nn.Sequential(LineCurveBlock(128), LineCurveBlock(128))
        self.down2 = ConvBlock(128, 256, stride=2)

        self.mid = nn.Sequential(*[LineCurveBlock(256) for _ in range(4)])

        self.up2 = ConvBlock(256, 128)
        self.dec2 = nn.Sequential(ConvBlock(256, 128), LineCurveBlock(128))
        self.up1 = ConvBlock(128, 64)
        self.dec1 = nn.Sequential(ConvBlock(128, 64), LineCurveBlock(64))

        self.out = nn.Conv2d(64, 3, 7, padding=3)

    def forward(self, x):
        f0 = self.enc1(self.head(x))
        f1 = self.enc2(self.down1(f0))
        f2 = self.mid(self.down2(f1))

        u2 = F.interpolate(f2, scale_factor=2, mode="bilinear", align_corners=False)
        u2 = self.up2(u2)
        d2 = self.dec2(torch.cat([u2, f1], dim=1))

        u1 = F.interpolate(d2, scale_factor=2, mode="bilinear", align_corners=False)
        u1 = self.up1(u1)
        d1 = self.dec1(torch.cat([u1, f0], dim=1))

        return torch.clamp(self.out(d1) + x, -1, 1)

## Discriminator

In [ ]:
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()

        def block(in_ch, out_ch, stride, norm=True):
            layers = [nn.Conv2d(in_ch, out_ch, 4, stride, 1)]
            if norm:
                layers.append(nn.InstanceNorm2d(out_ch))
            layers.append(nn.LeakyReLU(0.2, True))
            return layers

        self.net = nn.Sequential(
            *block(3, 64, 2, norm=False),
            *block(64, 128, 2),
            *block(128, 256, 2),
            *block(256, 512, 1),
            nn.Conv2d(512, 1, 4, 1, 1),
        )

    def forward(self, x):
        return self.net(x)

## Loss functions and metrics (PSNR, SSIM, MAE)

In [ ]:
sobel_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=torch.float32)
sobel_y = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=torch.float32)
sobel_x = sobel_x.view(1, 1, 3, 3).repeat(3, 1, 1, 1).to(device)
sobel_y = sobel_y.view(1, 1, 3, 3).repeat(3, 1, 1, 1).to(device)


def edge_loss(fake, real):
    fx = F.conv2d(fake, sobel_x, padding=1, groups=3)
    fy = F.conv2d(fake, sobel_y, padding=1, groups=3)
    rx = F.conv2d(real, sobel_x, padding=1, groups=3)
    ry = F.conv2d(real, sobel_y, padding=1, groups=3)
    return F.l1_loss(fx, rx) + F.l1_loss(fy, ry)


vgg = models.vgg19(weights=models.VGG19_Weights.DEFAULT).features[:15].to(device).eval()
for p in vgg.parameters():
    p.requires_grad = False

vgg_mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
vgg_std = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)


def perceptual_loss(fake, real):
    fake = ((fake + 1) / 2 - vgg_mean) / vgg_std
    real = ((real + 1) / 2 - vgg_mean) / vgg_std
    return F.mse_loss(vgg(fake), vgg(real))


def ssim(fake, real, window=11, sigma=1.5):
    fake = (fake.clamp(-1, 1) + 1) / 2
    real = (real + 1) / 2
    coords = torch.arange(window, dtype=torch.float32, device=fake.device) - window // 2
    g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
    g = g / g.sum()
    kernel = (g[:, None] * g[None, :]).expand(3, 1, window, window).contiguous()
    mu_f = F.conv2d(fake, kernel, groups=3)
    mu_r = F.conv2d(real, kernel, groups=3)
    var_f = F.conv2d(fake * fake, kernel, groups=3) - mu_f ** 2
    var_r = F.conv2d(real * real, kernel, groups=3) - mu_r ** 2
    cov = F.conv2d(fake * real, kernel, groups=3) - mu_f * mu_r
    c1 = 0.01 ** 2
    c2 = 0.03 ** 2
    score = ((2 * mu_f * mu_r + c1) * (2 * cov + c2)) / ((mu_f ** 2 + mu_r ** 2 + c1) * (var_f + var_r + c2))
    return score.mean(dim=(1, 2, 3))


def mae(fake, real):
    return ((fake.clamp(-1, 1) - real).abs() * 127.5).mean(dim=(1, 2, 3))


def evaluate(net, loader):
    net.eval()
    p_all, s_all, m_all = [], [], []
    with torch.no_grad():
        for blur, clean in loader:
            blur = blur.to(device)
            clean = clean.to(device)
            fake = net(blur)
            if isinstance(fake, (list, tuple)):
                fake = fake[0]
            fake = fake.clamp(-1, 1)
            mse = (((fake + 1) / 2 - (clean + 1) / 2) ** 2).mean(dim=(1, 2, 3))
            p_all.append(10 * torch.log10(1 / (mse + 1e-8)))
            s_all.append(ssim(fake, clean))
            m_all.append(mae(fake, clean))
    return torch.cat(p_all), torch.cat(s_all), torch.cat(m_all)

## Model, optimizers, loss weights

In [ ]:
G = Generator().to(device)
D = Discriminator().to(device)

opt_g = torch.optim.Adam(G.parameters(), lr=LR, betas=(0.5, 0.999))
opt_d = torch.optim.Adam(D.parameters(), lr=LR, betas=(0.5, 0.999))

W_L1 = 1.0
W_EDGE = 1.0
W_PERC = 0.1
W_ADV = 0.01

print("generator params:", sum(p.numel() for p in G.parameters()) / 1e6, "M")
print("discriminator params:", sum(p.numel() for p in D.parameters()) / 1e6, "M")

## Blurry input metrics (reference before training)

In [ ]:
p, s, m = evaluate(nn.Identity(), val_loader)
print(f"Blurry input | PSNR {p.mean():.2f} | SSIM {s.mean():.4f} | MAE {m.mean():.2f}")

## Checkpoint setup

In [ ]:
CKPT_DIR = "checkpoints_modified_gan"
os.makedirs(CKPT_DIR, exist_ok=True)
LAST_PATH = os.path.join(CKPT_DIR, "last.pth")
BEST_PATH = os.path.join(CKPT_DIR, "best.pth")

start_epoch = 0
best_psnr = 0.0
history = {"g_loss": [], "d_loss": [], "val_psnr": [], "val_ssim": [], "val_mae": []}

RESUME = False

if RESUME and os.path.exists(LAST_PATH):
    ckpt = torch.load(LAST_PATH, map_location=device)
    G.load_state_dict(ckpt["G"])
    D.load_state_dict(ckpt["D"])
    opt_g.load_state_dict(ckpt["opt_g"])
    opt_d.load_state_dict(ckpt["opt_d"])
    history = ckpt["history"]
    best_psnr = ckpt["best_psnr"]
    start_epoch = ckpt["epoch"] + 1
    print("resumed from epoch", start_epoch)

## Training

In [ ]:
for epoch in range(start_epoch, EPOCHS):
    G.train()
    D.train()
    g_sum = 0
    d_sum = 0

    for blur, clean in tqdm(train_loader, desc=f"Epoch {epoch + 1}/{EPOCHS}"):
        blur = blur.to(device, non_blocking=True)
        clean = clean.to(device, non_blocking=True)

        fake = G(blur)

        opt_d.zero_grad()
        d_real = D(clean)
        d_fake = D(fake.detach())
        d_loss = 0.5 * (F.mse_loss(d_real, torch.ones_like(d_real)) + F.mse_loss(d_fake, torch.zeros_like(d_fake)))
        d_loss.backward()
        opt_d.step()

        opt_g.zero_grad()
        d_out = D(fake)
        adv = F.mse_loss(d_out, torch.ones_like(d_out))
        l1 = F.l1_loss(fake, clean)
        edge = edge_loss(fake, clean)
        perc = perceptual_loss(fake, clean)
        g_loss = W_L1 * l1 + W_EDGE * edge + W_PERC * perc + W_ADV * adv
        g_loss.backward()
        opt_g.step()

        g_sum += g_loss.item()
        d_sum += d_loss.item()

    p, s, m = evaluate(G, val_loader)
    val_psnr = p.mean().item()
    val_ssim = s.mean().item()
    val_mae = m.mean().item()

    history["g_loss"].append(g_sum / len(train_loader))
    history["d_loss"].append(d_sum / len(train_loader))
    history["val_psnr"].append(val_psnr)
    history["val_ssim"].append(val_ssim)
    history["val_mae"].append(val_mae)

    print(f"epoch {epoch + 1} | G loss {history['g_loss'][-1]:.4f} | D loss {history['d_loss'][-1]:.4f} | val PSNR {val_psnr:.2f} | SSIM {val_ssim:.4f} | MAE {val_mae:.2f}")

    if history["val_psnr"][-1] > best_psnr:
        best_psnr = history["val_psnr"][-1]
        torch.save(G.state_dict(), BEST_PATH)

    torch.save({
        "G": G.state_dict(),
        "D": D.state_dict(),
        "opt_g": opt_g.state_dict(),
        "opt_d": opt_d.state_dict(),
        "history": history,
        "best_psnr": best_psnr,
        "epoch": epoch,
    }, LAST_PATH)

## Training curves

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
axes[0].plot(history["g_loss"], label="generator")
axes[0].plot(history["d_loss"], label="discriminator")
axes[0].set_title("Loss")
axes[0].legend()
axes[1].plot(history["val_psnr"])
axes[1].set_title("Validation PSNR")
axes[2].plot(history["val_ssim"])
axes[2].set_title("Validation SSIM")
axes[3].plot(history["val_mae"])
axes[3].set_title("Validation MAE")
for ax in axes:
    ax.set_xlabel("epoch")
plt.show()

## Visual results

In [ ]:
G.eval()
blur, clean = next(iter(val_loader))
with torch.no_grad():
    fake = G(blur.to(device)).cpu()


def show(t):
    return ((t.permute(1, 2, 0) + 1) / 2).clamp(0, 1).numpy()


n = min(4, blur.size(0))
fig, axes = plt.subplots(n, 3, figsize=(12, 3 * n), squeeze=False)
for i in range(n):
    axes[i][0].imshow(show(blur[i]))
    axes[i][1].imshow(show(fake[i]))
    axes[i][2].imshow(show(clean[i]))
    for j in range(3):
        axes[i][j].axis("off")
axes[0][0].set_title("Blurry")
axes[0][1].set_title("Deblurred")
axes[0][2].set_title("Clean")
plt.show()

Final evaluation with the best weights

In [ ]:
G.load_state_dict(torch.load(BEST_PATH, map_location=device))
p, s, m = evaluate(G, val_loader)
p, s, m = p.cpu().numpy(), s.cpu().numpy(), m.cpu().numpy()

is_motion = np.array([os.path.dirname(b) == MOTION_DIR for b, _ in val_pairs])
groups = {"all": np.ones(len(p), dtype=bool), "gaussian": ~is_motion, "motion": is_motion}

for name, mask in groups.items():
    print(f"{name:9s} | PSNR {p[mask].mean():.2f} | SSIM {s[mask].mean():.4f} | MAE {m[mask].mean():.2f}")